# 07 — Profile antibiotic vocabulary

## Goal

Before I write phenotype rules or a reporting-pair seed, I want to know exactly which antibiotic labels exist in ARMD and how much usable AST they carry.

The first few queries are source-level coverage checks on reviewed reporting organisms. They are not the final patient-deduplicated antibiogram denominator; they are just a way to see which drugs are actually represented in the data.

In [ ]:
%sql
select
    r.antibiotic,
    count(*) as endpoint_groups,
    count(distinct m.culture_organism_key) as culture_organisms,
    count(distinct m.patient_key) as patients

from workspace.micro_dev.int_mapped_cohort_culture_organism m

inner join workspace.micro_dev.int_susceptibility_reconciliation r
    on m.culture_organism_key = r.culture_organism_key

where m.include_primary_summary = 1
  and m.mapping_status = 'reviewed'
  and r.antibiotic is not null

group by r.antibiotic

order by endpoint_groups desc, r.antibiotic;

### Overall antibiotic volume

This gives me the high-level testing vocabulary and how often each antibiotic appears across reviewed culture-organism records.

In [ ]:
%sql
with eligible as (

    select
        cohort_id,
        organism_reporting_group,
        count(distinct culture_organism_key) as eligible_source_records

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where include_primary_summary = 1
      and mapping_status = 'reviewed'

    group by
        cohort_id,
        organism_reporting_group

),

endpoints as (

    select
        m.cohort_id,
        m.organism_reporting_group,
        r.antibiotic,

        count(*) as endpoint_groups,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Resistant',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as categorized_count,

        sum(
            case
                when r.resolution_status = 'conflict'
                    then 1
                else 0
            end
        ) as conflict_result_count,

        sum(
            case
                when r.resolution_status = 'missing'
                    then 1
                else 0
            end
        ) as missing_result_count

    from workspace.micro_dev.int_mapped_cohort_culture_organism m

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on m.culture_organism_key = r.culture_organism_key

    where m.include_primary_summary = 1
      and m.mapping_status = 'reviewed'
      and r.antibiotic is not null

    group by
        m.cohort_id,
        m.organism_reporting_group,
        r.antibiotic

)

select
    e.organism_reporting_group,
    p.antibiotic,
    e.eligible_source_records,
    p.endpoint_groups,
    e.eligible_source_records - p.endpoint_groups as untested_source_records,
    p.categorized_count,
    p.conflict_result_count,
    p.missing_result_count,

    round(
        100.0 * p.categorized_count / e.eligible_source_records,
        1
    ) as categorized_result_coverage_pct

from endpoints p

inner join eligible e
    on p.cohort_id = e.cohort_id
   and p.organism_reporting_group = e.organism_reporting_group

where p.cohort_id = 'blood'

order by
    e.organism_reporting_group,
    p.categorized_count desc,
    p.antibiotic;

### Blood-cohort coverage by reporting organism

For each organism-antibiotic combination I compare the number of reviewed source records with the number that have a categorized S/R/I result. This is useful for panel design, but I still review the final reporting pairs manually.

In [ ]:
%sql
with eligible as (

    select
        organism_reporting_group,
        count(distinct culture_organism_key) as eligible_source_records

    from workspace.micro_dev.int_mapped_cohort_culture_organism

    where cohort_id = 'blood'
      and include_primary_summary = 1
      and mapping_status = 'reviewed'

    group by organism_reporting_group

),

pair_counts as (

    select
        m.organism_reporting_group,
        r.antibiotic,

        sum(
            case
                when r.resolution_status = 'resolved'
                 and r.susceptibility in (
                     'Susceptible',
                     'Resistant',
                     'Intermediate'
                 )
                    then 1
                else 0
            end
        ) as categorized_count

    from workspace.micro_dev.int_mapped_cohort_culture_organism m

    inner join workspace.micro_dev.int_susceptibility_reconciliation r
        on m.culture_organism_key = r.culture_organism_key

    where m.cohort_id = 'blood'
      and m.include_primary_summary = 1
      and m.mapping_status = 'reviewed'
      and r.antibiotic is not null

    group by
        m.organism_reporting_group,
        r.antibiotic

),

ranked as (

    select
        p.organism_reporting_group,
        p.antibiotic,
        e.eligible_source_records,
        p.categorized_count,

        round(
            100.0 * p.categorized_count / e.eligible_source_records,
            1
        ) as categorized_result_coverage_pct,

        row_number() over (
            partition by p.organism_reporting_group
            order by
                p.categorized_count desc,
                p.antibiotic
        ) as antibiotic_rank

    from pair_counts p

    inner join eligible e
        on p.organism_reporting_group = e.organism_reporting_group

)

select *
from ranked
where antibiotic_rank <= 10

order by
    organism_reporting_group,
    antibiotic_rank;

### Candidate drugs by coverage

Ranking by categorized-result count is only a profiling shortcut. It helps surface the common drugs for each organism; it does not automatically decide what belongs in the reporting seed.

In [4]:
%sql
select
    lower(trim(antibiotic)) as normalized_label,
    count(distinct antibiotic) as source_label_count,
    array_sort(collect_set(antibiotic)) as source_labels

from workspace.micro_dev.int_susceptibility_reconciliation

where antibiotic is not null

group by lower(trim(antibiotic))

having count(distinct antibiotic) > 1

order by source_label_count desc, normalized_label;

,normalized_label,source_label_count,source_labels


### Simple formatting variants

First I check for labels that differ only by case or surrounding whitespace. Ideally this returns no rows.

In [5]:
%sql
with vocabulary as (

    select
        antibiotic,

        lower(
            trim(antibiotic)
        ) as case_space_normalized_label,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as punctuation_insensitive_key,

        count(*) as endpoint_groups,

        count(
            distinct culture_organism_key
        ) as culture_organisms

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    antibiotic,
    case_space_normalized_label,
    punctuation_insensitive_key,
    endpoint_groups,
    culture_organisms

from vocabulary

order by antibiotic

,antibiotic,case_space_normalized_label,punctuation_insensitive_key,endpoint_groups,culture_organisms
0,Amikacin,amikacin,amikacin,68148,68148
1,Amoxicillin/Clavulanic Acid,amoxicillin/clavulanic acid,amoxicillinclavulanicacid,57645,57645
2,Ampicillin,ampicillin,ampicillin,93245,93245
3,Ampicillin/Sulbactam,ampicillin/sulbactam,ampicillinsulbactam,21826,21826
4,Aztreonam,aztreonam,aztreonam,24755,24755
5,Cefazolin,cefazolin,cefazolin,82603,82603
6,Cefepime,cefepime,cefepime,37779,37779
7,Cefiderocol,cefiderocol,cefiderocol,20,20
8,Cefotaxime,cefotaxime,cefotaxime,765,765
9,Cefotetan,cefotetan,cefotetan,667,667


### Full normalized vocabulary

I also build a punctuation-insensitive comparison key. This is for QA only; the actual source label stays intact.

In [6]:
%sql
with vocabulary as (

    select
        antibiotic,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as punctuation_insensitive_key,

        count(*) as endpoint_groups

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    punctuation_insensitive_key,

    count(
        distinct antibiotic
    ) as source_label_count,

    array_sort(
        collect_set(antibiotic)
    ) as source_labels,

    sum(endpoint_groups)
        as endpoint_groups

from vocabulary

group by punctuation_insensitive_key

having count(distinct antibiotic) > 1

order by
    source_label_count desc,
    punctuation_insensitive_key

,punctuation_insensitive_key,source_label_count,source_labels,endpoint_groups


### Candidate duplicate labels

Any rows here would need manual review before I merge terminology. Similar punctuation does not guarantee that two drug names should be treated as the same analytical endpoint.

In [4]:
%sql
with expected_antibiotics as (

    select *
    from values

        -- CRE phenotype
        ('CRE', 'Ertapenem'),
        ('CRE', 'Imipenem'),
        ('CRE', 'Meropenem'),
        ('CRE', 'Meropenem/Vaborbactam'),
        ('CRE', 'Imipenem/Relebactam'),

        -- Oxacillin-resistant Staphylococcus aureus
        ('OXACILLIN_RESISTANT_S_AUREUS', 'Oxacillin'),
        ('OXACILLIN_RESISTANT_S_AUREUS', 'Cefoxitin'),

        -- Vancomycin-resistant Enterococcus
        ('VRE', 'Vancomycin'),

        -- MDR Pseudomonas aeruginosa:
        -- extended-spectrum cephalosporins
        ('MDR_P_AERUGINOSA', 'Cefepime'),
        ('MDR_P_AERUGINOSA', 'Ceftazidime'),
        ('MDR_P_AERUGINOSA', 'Ceftazidime/Avibactam'),
        ('MDR_P_AERUGINOSA', 'Ceftolozane/Tazobactam'),

        -- fluoroquinolones
        ('MDR_P_AERUGINOSA', 'Ciprofloxacin'),
        ('MDR_P_AERUGINOSA', 'Levofloxacin'),

        -- aminoglycosides
        ('MDR_P_AERUGINOSA', 'Tobramycin'),

        -- carbapenems
        ('MDR_P_AERUGINOSA', 'Imipenem'),
        ('MDR_P_AERUGINOSA', 'Meropenem'),
        ('MDR_P_AERUGINOSA', 'Imipenem/Relebactam'),

        -- antipseudomonal penicillin combination
        ('MDR_P_AERUGINOSA', 'Piperacillin/Tazobactam'),

        -- siderophore cephalosporin
        ('MDR_P_AERUGINOSA', 'Cefiderocol'),
        
        -- NHSN extended-spectrum cephalosporin-resistant
        -- Enterobacterales phenotype
        ('ESCE_ENTEROBACTERALES', 'Cefepime'),
        ('ESCE_ENTEROBACTERALES', 'Ceftriaxone'),
        ('ESCE_ENTEROBACTERALES', 'Cefotaxime'),
        ('ESCE_ENTEROBACTERALES', 'Ceftazidime'),
        ('ESCE_ENTEROBACTERALES', 'Ceftazidime/Avibactam'),
        ('ESCE_ENTEROBACTERALES', 'Ceftolozane/Tazobactam')
        

    as expected(
        phenotype_rule,
        expected_antibiotic
    )

),

expected_normalized as (

    select
        phenotype_rule,
        expected_antibiotic,

        regexp_replace(
            lower(trim(expected_antibiotic)),
            '[^a-z0-9]',
            ''
        ) as comparison_key

    from expected_antibiotics

),

source_vocabulary as (

    select
        antibiotic,

        regexp_replace(
            lower(trim(antibiotic)),
            '[^a-z0-9]',
            ''
        ) as comparison_key,

        count(*) as endpoint_groups,

        count(
            distinct culture_organism_key
        ) as culture_organisms

    from workspace.micro_dev.int_susceptibility_reconciliation

    where antibiotic is not null

    group by antibiotic

)

select
    e.phenotype_rule,
    e.expected_antibiotic,

    count(
        distinct v.antibiotic
    ) as matched_source_label_count,

    array_sort(
        collect_set(v.antibiotic)
    ) as matched_source_labels,

    coalesce(
        sum(v.endpoint_groups),
        0
    ) as endpoint_groups,

    coalesce(
        sum(v.culture_organisms),
        0
    ) as culture_organisms,

    case
        when count(distinct v.antibiotic) = 0
            then 'not_present'

        when count(distinct v.antibiotic) = 1
            then 'single_source_label'

        else 'multiple_candidate_labels'
    end as vocabulary_status

from expected_normalized e

left join source_vocabulary v
    on e.comparison_key = v.comparison_key

group by
    e.phenotype_rule,
    e.expected_antibiotic

order by
    e.phenotype_rule,
    e.expected_antibiotic

,phenotype_rule,expected_antibiotic,matched_source_label_count,matched_source_labels,endpoint_groups,culture_organisms,vocabulary_status
0,CRE,Ertapenem,1,[Ertapenem],57088,57088,single_source_label
1,CRE,Imipenem,1,[Imipenem],12171,12171,single_source_label
2,CRE,Imipenem/Relebactam,0,[],0,0,not_present
3,CRE,Meropenem,1,[Meropenem],70151,70151,single_source_label
4,CRE,Meropenem/Vaborbactam,1,[Meropenem/Vaborbactam],26,26,single_source_label
5,ESCE_ENTEROBACTERALES,Cefepime,1,[Cefepime],37779,37779,single_source_label
6,ESCE_ENTEROBACTERALES,Cefotaxime,1,[Cefotaxime],765,765,single_source_label
7,ESCE_ENTEROBACTERALES,Ceftazidime,1,[Ceftazidime],68473,68473,single_source_label
8,ESCE_ENTEROBACTERALES,Ceftazidime/Avibactam,1,[Ceftazidime/Avibactam],1488,1488,single_source_label
9,ESCE_ENTEROBACTERALES,Ceftolozane/Tazobactam,1,[Ceftolozane/Tazobactam],3854,3854,single_source_label


### Drugs required by phenotype rules

This is the final vocabulary check before phenotype modeling. I compare every antibiotic referenced by the planned CRE, ESCE-R, oxacillin-resistance, VRE, and MDR *P. aeruginosa* rules with the source vocabulary.

A drug can be absent and still remain part of the rule definition; absence just means that ARMD cannot contribute evidence for that endpoint in this release.

### Takeaway

The antibiotic vocabulary is explicit enough to move into phenotype modeling. From here on, phenotype logic uses reviewed source labels and reconciled susceptibility states rather than fuzzy drug-name matching.